# מפגש 15 — פתרון מורה מלא עם בדיקות
מטלת המקור P1–P5, שאילתה (2,2), מרחק בין (2,1) ל־(5,5). זו מימוש ידני ב־Pandas, **ללא** Scikit-learn classifier וללא הערכת Accuracy.

In [ ]:
import math
import numpy as np
import pandas as pd
distance_manual=math.sqrt((5-2)**2+(5-1)**2)
reverse_distance=math.sqrt((2-5)**2+(1-5)**2)
print('Distance and symmetry:',distance_manual,reverse_distance)
assert distance_manual==5
assert reverse_distance==5

## קלטים, תוויות ותצפית חדשה
שומרים עותק מקור כדי להוכיח שהמרחק חושב ממאפיינים בלבד ולא נשמר כשדה אימון.

In [ ]:
training_df=pd.DataFrame({
'id':['P1','P2','P3','P4','P5'],
'f1':[2,4,4,6,2],
'f2':[1,2,4,2,7],
'label':['B','A','A','B','A']})
query_f1,query_f2=2,2
original=training_df.copy(deep=True)
assert training_df.shape==(5,4)
assert training_df['label'].tolist()==['B','A','A','B','A']

## מרחק, דירוג וטבלת פתרון
P1=1,P2=2,P3=√8,P4=4,P5=5 — המרחקים שונים ולכן אין תיקו בגבול מספר השכנים.

In [ ]:
distance_df=training_df.copy(deep=True)
distance_df['dx']=distance_df['f1']-query_f1
distance_df['dy']=distance_df['f2']-query_f2
distance_df['squared']=distance_df['dx']**2+distance_df['dy']**2
distance_df['distance']=distance_df['squared']**0.5
ranked_df=distance_df.sort_values('distance',kind='stable')
display(ranked_df[['id','dx','dy','squared','distance','label']])
assert ranked_df['id'].tolist()==['P1','P2','P3','P4','P5']
assert ranked_df['squared'].tolist()==[1,4,8,16,25]
assert np.allclose(ranked_df['distance'],[1,2,np.sqrt(8),4,5])
assert ranked_df['label'].tolist()==['B','A','A','B','A']

## הצבעה ברורה ללא כלל הכרעת תיקו נסתר
בנתוני המטלה אין תיקו בהצבעות K 1,3,5. המסקנה המילולית נדרשת מעבר לקוד.

In [ ]:
outcomes={1:'B',3:'A',5:'A'}
for k in [1,3,5]:
    near=ranked_df.head(k)
    vote_counts=near['label'].value_counts().to_dict()
    modes=near['label'].mode().tolist()
    print('K',k,'IDs',near['id'].tolist(),'votes',vote_counts,'predictions',modes)
    assert modes==[outcomes[k]]
assert ranked_df.head(3)['id'].tolist()==['P1','P2','P3']
assert ranked_df.head(3)['label'].tolist()==['B','A','A']
assert ranked_df.head(5)['label'].value_counts().to_dict()=={'A':3,'B':2}

## שמירת מקור, גבול K ותיקו רב־מחלקתי
לשאילתה אין y אמת. אי אפשר לחשב Accuracy ממנה; K חייב להיות בין 1 ל־5. תיקו של A,B,C מדגים ש־K אי־זוגי אינו מבטיח רוב כשיש יותר משתי מחלקות.

In [ ]:
assert training_df.equals(original)
assert 'distance' not in training_df.columns
assert 'distance' in distance_df.columns
assert len(training_df)==5
assert 7>len(training_df)
tie=pd.Series(['A','B','C']).mode().tolist()
print('Tie modes:',tie)
assert tie==['A','B','C']
assert len(tie)==3

## מציגים גרף שניתן לבדוק מול הטבלה
יחס צירים שווה; קווים לשלושת השכנים הקרובים, מסומנים עם מזהים. אין תווית אמת לשאילתה.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,5))
for label,color in [('A','royalblue'),('B','firebrick')]:
    part=training_df[training_df['label']==label]
    ax.scatter(part['f1'],part['f2'],s=100,color=color,label=label)
for i in range(len(training_df)):
    row=training_df.iloc[i]
    ax.annotate(row['id'],(row['f1']+.1,row['f2']+.1))
for i in range(3):
    row=ranked_df.iloc[i]
    ax.plot([query_f1,row['f1']],[query_f2,row['f2']],linestyle='--',color='gray')
ax.scatter([query_f1],[query_f2],marker='*',s=220,color='black',label='Unknown query')
ax.set_xlabel('f1 (same teaching scale)')
ax.set_ylabel('f2 (same teaching scale)')
ax.set_title('Student exercise - 3 nearest neighbors')
ax.set_aspect('equal',adjustable='box')
ax.legend()
plt.tight_layout()
plt.show()

## תיעוד מורה
K1=B, K3=A ברוב 2:1, K5=A ברוב 3:2. אין אמת ידועה, לכן אין Accuracy. רשמו אסטרטגיית Scaling כשיחידות שונות; כל Fit בהכנה נלמד מתוך Train בלבד. מימוש KNN בפונקציה יופיע במפגש 16.